# Week 3: Pandas Data Cleaning Demo

**Learning Goals:**
- Load and inspect messy historical data
- Identify common data quality issues
- Apply systematic cleaning techniques
- Make informed decisions about data transformation
- Document cleaning choices

## Setup

In [223]:
import pandas as pd
import re

# Display settings for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## Step 1: Load and Inspect Data

First, let's load our historical dataset and see what we're working with.

In [224]:
# Load the data
df = pd.read_csv('./week03-census-sample.csv')

# Display all rows with all columns side by side
print("All rows:")
df

All rows:


,Name,Birth Year,Location,Occupation,Notes
0,Frederick Douglass,1818,Talbot County MD,abolitionist writer,escaped slavery
1,Harriet Tubman,c. 1822,Dorchester County MD,conductor,Underground Railroad
2,Sojourner Truth,1797?,New York,Preacher and activist,NaN
3,S. Truth,1797,new york,preacher,duplicate?
4,Frederick douglass,1818,Talbot County,Writer,possible duplicate
5,Ida B. Wells,1862,Holly Springs MS,journalist,NaN
6,W.E.B. Du Bois,1868,Great Barrington MA,sociologist historian,NaN
7,Booker T. Washington,1856?,Hale's Ford VA,educator,born enslaved
8,Mary Church Terrell,1863,Memphis TN,suffragist,NaN


Note that the dataset contains various inconsistencies and missing values that we will need to address. Note also that some cells have 'NaN' as a string, which is different from actual missing values represented by `NaN` in pandas.

In [225]:
# Check the shape (rows, columns)
print(df.shape)


(9, 5)


In [226]:
# Get column information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Name        9 non-null      str  
 1   Birth Year  9 non-null      str  
 2   Location    9 non-null      str  
 3   Occupation  9 non-null      str  
 4   Notes       5 non-null      str  
dtypes: str(5)
memory usage: 492.0 bytes


**Observations:**
- All columns are 'object' type (strings)
- An object column can contain any type of data, but here it means we have not yet converted numeric columns to the appropriate types
- Birth Year should probably be numeric
- Some columns might have missing values

## Step 2: Identify Data Quality Issues

Let's systematically check for problems in our data.

Let's check for whitespace issues in string columns.

In [227]:
# Check for whitespace issues in string columns using pandas

for col in df.select_dtypes(include=['object']).columns:
    # The select_dtypes() method is used to select only the columns of type 'object', which typically represent string data in a DataFrame. This ensures that we are only checking for whitespace issues in columns that contain string values, as leading or trailing whitespace is relevant for string data.
    if df[col].str.contains(r'^\s+|\s+$', na=False).any():
        print(f"Column '{col}' has leading or trailing whitespace.")
        
        #regex pattern r'^\s+|\s+$' checks for leading (^\s+) or trailing (\s+$) whitespace in the string values of the column
        # the str.contains() method is used to check if any of the values in the column match the regex pattern, and na=False ensures that NaN values are not considered as matches
        # The na=False argument in the str.contains() method is used to handle any missing values (NaN) in the column. By default, str.contains() returns NaN for any missing values, which can lead to issues when checking for matches. Setting na=False ensures that any missing values are treated as non-matches, allowing the method to return a boolean value (True or False) instead of NaN. This way, you can accurately identify if there are any leading or trailing whitespace issues in the string columns without being affected by missing data.
        # the any() method is used to check if there are any True values in the resulting boolean series, indicating that at least one value in the column has leading or trailing whitespace. If such a value is found, a message is printed indicating which column has the issue.
        # the f string is a way to format the output message. This allows you to easily identify which specific column has the whitespace issue by including the column name in the message.
        # Notice the use of curly braces {} to insert the column name into the message, making it clear which column has the whitespace issue.


Column 'Name' has leading or trailing whitespace.


C:\Users\bskopyk\AppData\Local\Temp\ipykernel_15380\3860948480.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [228]:
# show the rows with whitespace issues in the Name column
print(f"Rows with whitespace issues in '{col}' column:")
print(df[df['Name'].str.contains(r'^\s+|\s+$', na=False)])

# Count affected rows before cleaning
affected_rows = df['Name'].str.contains(r'^\s+|\s+$', na=False).sum()

# Remove whitespace
df['Name'] = df['Name'].str.strip()

# Verify removal
if not df['Name'].str.contains(r'^\s+|\s+$', na=False).any():
    print(f"Whitespace successfully removed from 'Name' column. Affected rows: {affected_rows}")
else:
    print(f"Some whitespace may still remain in 'Name' column.")
affected_rows = df['Name'].str.contains(r'^\s+|\s+$', na=False).sum()
# show the df before and after the change
df['Name'] = df['Name'].str.strip()
# Verify removal
if not df['Name'].str.contains(r'^\s+|\s+$', na=False).any():
    print(f"Whitespace successfully removed from 'Name' column. Affected rows: {affected_rows}")

Rows with whitespace issues in 'Notes' column:
                    Name Birth Year        Location Occupation          Notes
3              S. Truth        1797        new york   preacher     duplicate?
7   Booker T. Washington      1856?  Hale's Ford VA   educator  born enslaved
Whitespace successfully removed from 'Name' column. Affected rows: 2
Whitespace successfully removed from 'Name' column. Affected rows: 0


In [229]:
# Check for missing values
print("Missing values per column:")
df.isnull().sum()

Missing values per column:


Name          0
Birth Year    0
Location      0
Occupation    0
Notes         4
dtype: int64

In [230]:
df["Notes"]

0         escaped slavery
1    Underground Railroad
2                     NaN
3              duplicate?
4      possible duplicate
5                     NaN
6                     NaN
7           born enslaved
8                     NaN
Name: Notes, dtype: str

In [231]:
# Examine Birth Year values
# The unique() method is used to retrieve all the distinct birth years present in the dataset
print("Unique Birth Year values:")
print(df['Birth Year'].unique())

Unique Birth Year values:
<StringArray>
['1818', 'c. 1822', '1797?', '1797', '1862', '1868', '1856?', '1863']
Length: 8, dtype: str


In [232]:
# Check Location formatting
print("Unique Location values:".upper() + "\n")
# Put values in an alphabetical vertical list for easier reading
print("\n".join(sorted(df['Location'].unique())))
# The join method is used to concatenate the unique location values into a single string, with each value separated by a newline character (\n)
# notice how the lowercase 'new york' and uppercase 'NEW YORK' are treated as distinct values

UNIQUE LOCATION VALUES:

Dorchester County  MD
Great Barrington MA
Hale's Ford VA
Holly Springs MS
Memphis TN
New York
Talbot County
Talbot County MD
new york


In [233]:
# Look for potential duplicates
print("ALL NAMES:" + "\n--------------------")
print(df['Name'])

# Let's also see these in a vertical list sorted alphabetically for easier reading
print("\nAll names sorted alphabetically:".upper() + "\n--------------------")
print("\n".join(sorted(df['Name'].values)))

ALL NAMES:
--------------------
0      Frederick Douglass
1          Harriet Tubman
2         Sojourner Truth
3                S. Truth
4      Frederick douglass
5            Ida B. Wells
6          W.E.B. Du Bois
7    Booker T. Washington
8     Mary Church Terrell
Name: Name, dtype: str

ALL NAMES SORTED ALPHABETICALLY:
--------------------
Booker T. Washington
Frederick Douglass
Frederick douglass
Harriet Tubman
Ida B. Wells
Mary Church Terrell
S. Truth
Sojourner Truth
W.E.B. Du Bois


**Issues Identified:**
1. Birth Year contains uncertainty markers: "c. " (circa) and "?"
2. Location has inconsistent capitalization and extra spaces
3. Potential duplicate entries (Frederick Douglass, Sojourner Truth)
4. Missing value in Notes column

## Step 3: Clean Column Names

Make column names easier to work with: lowercase, underscores instead of spaces.

In [234]:
# Clean column names programmatically
df.columns = df.columns.str.lower().str.replace(' ', '_')

# df.columns is used to access the column names of the DataFrame
# The str.lower() method is applied to convert all column names to lowercase, ensuring consistency and avoiding issues with case sensitivity when referencing columns later in the code.
# The str.replace(' ', '_') method is used to replace any spaces in the column names with underscores, which is a common practice in programming to create more readable and consistent column names. This
    
print("New column names:\n--------------------")
print(df.columns.tolist())

New column names:
--------------------
['name', 'birth_year', 'location', 'occupation', 'notes']


In [235]:
# let's see the df again
df

,name,birth_year,location,occupation,notes
0,Frederick Douglass,1818,Talbot County MD,abolitionist writer,escaped slavery
1,Harriet Tubman,c. 1822,Dorchester County MD,conductor,Underground Railroad
2,Sojourner Truth,1797?,New York,Preacher and activist,NaN
3,S. Truth,1797,new york,preacher,duplicate?
4,Frederick douglass,1818,Talbot County,Writer,possible duplicate
5,Ida B. Wells,1862,Holly Springs MS,journalist,NaN
6,W.E.B. Du Bois,1868,Great Barrington MA,sociologist historian,NaN
7,Booker T. Washington,1856?,Hale's Ford VA,educator,born enslaved
8,Mary Church Terrell,1863,Memphis TN,suffragist,NaN


## Step 4: Clean Birth Year Column

Remove uncertainty markers and convert to numeric type.

**Historical Decision:** We're removing information about uncertainty. In a real project, consider creating a separate `birth_year_certain` boolean column!

In [236]:
# Let's create a certainty boolean column to flag years with with circa (c.) or ? in column
df['b_year_certain'] = ~df['birth_year'].astype(str).str.contains(r'c\.|\?')
# notice the use of the ~ operator to negate the boolean values returned by the str.contains() method, so that the 'certainty' column will be True for rows where the 'birth_year' does not contain 'c.' or '?', and False for rows where it does contain those characters.
# Now let's convert the 'birth_year' column to numeric, coercing any non-numeric values (like those with 'c.' or '?') to NaN
# for birth_year column, strip non-numeric characters
df['birth_year'] = (df['birth_year'].astype(str).str.replace(r'[^0-9]', '', regex=True)
)
# Convert to numeric, coercing errors to NaN and use nullable integer
df['birth_year'] = pd.to_numeric(df['birth_year'], errors='coerce').astype('Int64')
print(df.info())
df


<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   name            9 non-null      str  
 1   birth_year      9 non-null      Int64
 2   location        9 non-null      str  
 3   occupation      9 non-null      str  
 4   notes           5 non-null      str  
 5   b_year_certain  9 non-null      bool 
dtypes: Int64(1), bool(1), str(4)
memory usage: 510.0 bytes
None


,name,birth_year,location,occupation,notes,b_year_certain
0,Frederick Douglass,1818,Talbot County MD,abolitionist writer,escaped slavery,True
1,Harriet Tubman,1822,Dorchester County MD,conductor,Underground Railroad,False
2,Sojourner Truth,1797,New York,Preacher and activist,NaN,False
3,S. Truth,1797,new york,preacher,duplicate?,True
4,Frederick douglass,1818,Talbot County,Writer,possible duplicate,True
5,Ida B. Wells,1862,Holly Springs MS,journalist,NaN,True
6,W.E.B. Du Bois,1868,Great Barrington MA,sociologist historian,NaN,True
7,Booker T. Washington,1856,Hale's Ford VA,educator,born enslaved,False
8,Mary Church Terrell,1863,Memphis TN,suffragist,NaN,True


## Step 5: Standardize Location

Fix capitalization and remove extra whitespace.

In [237]:
# Standardize: title case, strip extra whitespace
# Note: .title() can incorrectly capitalize letters after apostrophes (e.g., "Hale's" -> "Hale'S")
# We'll fix this by converting to title case first, then lowercasing any letter after an apostrophe
df['location'] = (df['location']
    .str.strip()
    .str.title()
    .str.replace(r"'([A-Z])", lambda m: "'" + m.group(1).lower(), regex=True)
    # Collapse multiple internal spaces
    .str.replace(r'\s+', ' ', regex=True)
)
# The 'r' before the string in the str.replace() method indicates that the string is a raw string, which allows us to use backslashes without needing to escape them.
# A lambda function sets an object (m) and returns a string that consists of an apostrophe followed by the lowercase version of the captured uppercase letter (m.group(1)).

print("Standardized locations:".upper()+"\n----------------")
print("\n".join(sorted(df['location'])))

STANDARDIZED LOCATIONS:
----------------
Dorchester County Md
Great Barrington Ma
Hale's Ford Va
Holly Springs Ms
Memphis Tn
New York
New York
Talbot County
Talbot County Md


In [238]:
# Convert any standalone two-letter token (case-insensitive) to uppercase (e.g., "md" or "Md" -> "MD")

# Uppercase standalone two-letter tokens (case-insensitive)
df['location'] = df['location'].str.replace(
    r'\b([a-z]{2})\b',
    lambda m: m.group(1).upper(),
    regex=True,
    flags=re.IGNORECASE  # re.IGNORECASE makes the pattern case-insensitive (matches 'md','Md','MD', etc.)
)

# Show result
df["location"]

0        Talbot County MD
1    Dorchester County MD
2                New York
3                New York
4           Talbot County
5        Holly Springs MS
6     Great Barrington MA
7          Hale's Ford VA
8              Memphis TN
Name: location, dtype: str

Let's put the state info in a new column for easier analysis later.

In [239]:
# Put state info in a new column for easier analysis later
# Extract state abbreviation
df['state'] = df['location'].str.extract(r'\b([A-Z]{2})\b', expand=False)

# Remove state abbreviation from location
df['location'] = df['location'].str.replace(r'\s+\b[A-Z]{2}\b', '', regex=True).str.strip()

df[["location", "state"]]

,location,state
0,Talbot County,MD
1,Dorchester County,MD
2,New York,NaN
3,New York,NaN
4,Talbot County,NaN
5,Holly Springs,MS
6,Great Barrington,MA
7,Hale's Ford,VA
8,Memphis,TN


## Step 6: Handle Name Duplicates

This requires historical judgment! Let's investigate potential duplicates.

In [240]:
# split name into first and last name columns, handling middle names/initials
name_split = (df['name']
              .str.strip()
              .str.lower() # values in name column will be lower case
              .str.split(' ', n=2, expand=True) # n=2 means split into at most 3 parts
                                                # expand=True means the result will be a DataFrame with separate columns for each split part
              )

# split name into first and last name columns, handling middle names/initials
name_split = df['name'].str.strip().str.lower().str.split(' ', n=2, expand=True)
df['first_name'] = name_split[0]
df['last_name'] = name_split[2].combine_first(name_split[1])
df['middle_name'] = name_split[1].where(name_split[2].notna())

# If middle name is de, da, du, van, von, etc., move it to last name
particles = {'de','da','du','van','von'}
mask = df['middle_name'].notna() & df['middle_name'].str.strip().str.lower().isin(particles)
df.loc[mask, 'last_name'] = df.loc[mask, 'middle_name'].str.strip() + ' ' + df.loc[mask, 'last_name'].str.strip()
df.loc[mask, 'middle_name'] = pd.NA

# print df as string
print(df.to_string())

                   name  birth_year           location             occupation                 notes  b_year_certain state first_name   last_name middle_name
0    Frederick Douglass        1818      Talbot County    abolitionist writer       escaped slavery            True    MD  frederick    douglass         NaN
1        Harriet Tubman        1822  Dorchester County              conductor  Underground Railroad           False    MD    harriet      tubman         NaN
2       Sojourner Truth        1797           New York  Preacher and activist                   NaN           False   NaN  sojourner       truth         NaN
3              S. Truth        1797           New York               preacher            duplicate?            True   NaN         s.       truth         NaN
4    Frederick douglass        1818      Talbot County                 Writer    possible duplicate            True   NaN  frederick    douglass         NaN
5          Ida B. Wells        1862      Holly Springs    

In [241]:
# Find rows that share any first_name, middle_name, or last_name value
# (excluding NaN/null/empty values)
from collections import defaultdict

name_columns = ['first_name', 'middle_name', 'last_name']
value_to_indices = defaultdict(set)

# Build a mapping of each name value to the row indices that have it
for col in name_columns:
    s = df[col]
    non_blank = s.notna() & (s.astype(str).str.strip() != '')
    
    for idx in df[non_blank].index:
        val = s[idx]
        if pd.notna(val) and str(val).strip():
            value_to_indices[val].add(idx)

# Find indices that share at least one name value
indices_with_shared_names = set()
for val, indices in value_to_indices.items():
    if len(indices) > 1:  # Value appears in multiple rows
        indices_with_shared_names.update(indices)

# Group rows by their shared connections
groups = defaultdict(set)
for val, indices in value_to_indices.items():
    if len(indices) > 1:
        for idx in indices:
            groups[idx].update(indices)

# Consolidate overlapping groups using union-find
def merge_groups(groups):
    """Merge groups that share any indices"""
    merged = []
    remaining = list(groups.values())
    
    while remaining:
        current = remaining.pop(0)
        changed = True
        while changed:
            changed = False
            new_remaining = []
            for other in remaining:
                if current & other:  # If groups overlap
                    current |= other
                    changed = True
                else:
                    new_remaining.append(other)
            remaining = new_remaining
        merged.append(current)
    
    return merged

consolidated = merge_groups(groups)

# Display results
print(f"Found {len(consolidated)} group(s) of rows sharing name values:\n")
for i, group in enumerate(consolidated, 1):
    sorted_indices = sorted(group)
    print(f"Group {i}: rows {sorted_indices}")
    display(df.loc[sorted_indices, ['name', 'first_name', 'middle_name', 'last_name', 'birth_year', 'location']].sort_values('name'))
    print()

Found 2 group(s) of rows sharing name values:

Group 1: rows [0, 4]


,name,first_name,middle_name,last_name,birth_year,location
0,Frederick Douglass,frederick,NaN,douglass,1818,Talbot County
4,Frederick douglass,frederick,NaN,douglass,1818,Talbot County



Group 2: rows [2, 3]


,name,first_name,middle_name,last_name,birth_year,location
3,S. Truth,s.,NaN,truth,1797,New York
2,Sojourner Truth,sojourner,NaN,truth,1797,New York


**Discussion Questions:**
- Are these the same person?
- Which record should we keep?
- What information would we lose by removing duplicates?

**Our Decision:** Remove entries that appear to be duplicates (rows 3 and 4)

In [243]:
# We need to assess the results and determine if we want to remove any of these rows as duplicates, or if they represent different individuals with shared name values.
# We can safely say that the rows with last_name "truth" and last_name "douglass" represent the same individuals. So we can remove one of the truth and one of the douglass.
# We might prefer to keep 0 and not 4; and to keep 2 and not 3.
# Let's remove rows 3 and 4, which are duplicates of rows 0 and 2, respectively.

df = df.drop(index=[3, 4]).reset_index(drop=True) # reset_index(drop=True) is used to reset the index of the DataFrame after dropping rows
df[["name", "birth_year", "location", "state"]]


,name,birth_year,location,state
0,Frederick Douglass,1818,Talbot County,MD
1,Harriet Tubman,1822,Dorchester County,MD
2,Sojourner Truth,1797,New York,NaN
3,Booker T. Washington,1856,Hale's Ford,VA
4,Mary Church Terrell,1863,Memphis,TN


## Step 7: Export Cleaned Data

Save the cleaned dataset and document what we did.

In [244]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   name            5 non-null      str  
 1   birth_year      5 non-null      Int64
 2   location        5 non-null      str  
 3   occupation      5 non-null      str  
 4   notes           3 non-null      str  
 5   b_year_certain  5 non-null      bool 
 6   state           4 non-null      str  
 7   first_name      5 non-null      str  
 8   last_name       5 non-null      str  
 9   middle_name     2 non-null      str  
dtypes: Int64(1), bool(1), str(8)
memory usage: 502.0 bytes


In [245]:
# Drop the temporary helper columns; but let's keep the state column for analysis; safe: ignore if missing
df = df.drop(columns=['first_name', 'last_name', 'middle_name'], errors='ignore')

# Export to CSV
df.to_csv('./week03-census-cleaned.csv', index=False)
print("Cleaned data saved to: ./week03-census-cleaned.csv")

Cleaned data saved to: ./week03-census-cleaned.csv


## Summary

### What We Did

1. ✅ Loaded and inspected the data
2. ✅ Identified data quality issues
3. ✅ Cleaned column names (lowercase, underscores)
4. ✅ Cleaned birth year values (removed uncertainty markers)
5. ✅ Standardized location formatting
6. ✅ Extracted state information into a new column
7. ✅ Identified and removed duplicate records
8. ✅ Exported cleaned data

### Important Takeaways

- **Cleaning is interpretation**: Every choice removes or transforms information
- **Document everything**: What changed? Why? What was lost?
- **Preserve original data**: Always keep raw data untouched
- **80% of data work is cleaning**: This is normal!
- **Use domain expertise**: Historical knowledge guides cleaning decisions

### Next Steps

1. Do the Programming Historian assignment
2. Apply these techniques to another dataset
3. Create a `README.md` documenting your cleaning steps
4. Keep an `AI-use-log.md` if you use Copilot
5. Consider what information your cleaning removes
